### Documents Structture

In [1]:
### Document Structure
from langchain_core.documents import Document

In [2]:

doc = Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source": "example.txt",
        "pages": 1,
        "author": "Krish Naik",
        "date_created": "2025-01-01"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Krish Naik', 'date_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [3]:
### Create a simple txt file 
import os 
os.makedirs("../data/text_files",exist_ok=True)

In [4]:
sample_texts = {
    "../data/text_files/cybersecurity_foundations.txt": """Cybersecurity Foundations

Cybersecurity is the practice of protecting systems, networks, and data from digital attacks.
As organizations become increasingly dependent on technology, securing digital assets
has become a critical priority across all industries.

Key Features:
- Threat detection and prevention
- Network security and monitoring
- Data encryption and protection
- Identity and access management
- Incident response strategies

Cybersecurity is widely applied in finance, healthcare, government, and enterprise
environments to ensure confidentiality, integrity, and availability of information.""",

    "../data/text_files/cloud_infrastructure_overview.txt": """Cloud Infrastructure Overview

Cloud infrastructure refers to the collection of hardware and software components
that enable cloud computing services. It allows organizations to deploy scalable,
flexible, and cost-efficient IT environments.

Key Features:
- Virtualization technologies
- On-demand resource provisioning
- High availability and redundancy
- Automated scaling
- Centralized management and monitoring

Cloud infrastructure is widely used in modern enterprises to support digital
transformation, application hosting, data storage, and disaster recovery strategies."""
}

In [5]:
for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created successfully!")

✅ Sample text files created successfully!


In [ ]:
from langchain_community.document_loaders import TextLoader

# Charger le deuxième fichier
loader = TextLoader("../data/text_files/cloud_infrastructure_overview.txt", encoding="utf-8")
document = loader.load()

# Charger le deuxième fichier
loader1 = TextLoader("../data/text_files/cybersecurity_foundations.txt", encoding="utf-8")
document1 = loader1.load()

print("\n=== Cloud Infrastructure Document ===")
print(document)

print("\n=== Foundation Document ===")


In [ ]:
from langchain_community.document_loaders import DirectoryLoader
DATA_DIR = "../data/text_files"

def load_txt_documents():
    loader = DirectoryLoader(
        path=DATA_DIR,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
        show_progress=True,
        use_multithreading=True,
    )
    return loader.load()

documents=load_txt_documents()

In [ ]:
import re
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = "../data/pdf_files"

def load_pdf_documents():
    loader = DirectoryLoader(
        path=DATA_DIR,
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress=True,
        use_multithreading=True,
    )
    return loader.load()

def clean_text(text: str) -> str:
    # Fix ligatures
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl").replace("\xa0", " ")

    # Normalize spaces (without removing paragraph breaks)
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive newlines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()

# 1) Load
documentspdf = load_pdf_documents()
print(f"Loaded {len(documentspdf)} documents")

In [9]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)
texts=documentspdf[0].page_content
texts




'SOP-IT-0003 — Backup Monitoring & Exception\nHandling\nDocument ID: SOP-IT-0003\nTitle: Backup Monitoring & Exception Handling\nVersion: 1.0\nStatus: Effective\nSite: Couvet\nDepartment: IT\nEffective Date: 2026-03-01\n1. Purpose\nDescribe how IT monitors backup jobs and handles exceptions to ensure continuous data\nprotection.\n2. Daily Monitoring\n- Review backup job status dashboard.\n- Check for failed or incomplete jobs.\n- Verify storage capacity warnings.\n3. Exception Handling\n- Create incident ticket.\n- Investigate root cause.\n- Re-run backup if possible.\n- Attach evidence to ticket.'

In [ ]:
# 2) Clean documents (before split)
for doc in documentspdf:
    doc.page_content = clean_text(doc.page_content)

# 3) Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)
chunks = text_splitter.split_documents(documentspdf)

# 4) Filter empty chunks + add useful metadata
filtered_chunks = []
for i, c in enumerate(chunks):
    if not c.page_content or not c.page_content.strip():
        continue
    c.metadata["chunk_index"] = i
    c.metadata["chunk_len"] = len(c.page_content)
    # (optionnel) normaliser le chemin source
    if "source" in c.metadata:
        c.metadata["source"] = str(c.metadata["source"])
    filtered_chunks.append(c)

print(f"Created {len(filtered_chunks)} chunks (filtered from {len(chunks)})")

# Preview
print("\n--- First chunk preview ---")
print(filtered_chunks[0].page_content[:500])
print("Metadata:", filtered_chunks[0].metadata)

In [109]:
# ---------------------------------------------------------------------------
# 0) Imports + ENV
# ---------------------------------------------------------------------------

import os
import re
import json
import time
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import chromadb
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
if not os.environ["OPENAI_API_KEY"]:
    raise RuntimeError("OPENAI_API_KEY is missing. Put it in your .env file.")

In [110]:
# ---------------------------------------------------------------------------
# 1) Embedding manager (OpenAI)
# ---------------------------------------------------------------------------

class EmbeddingManager:
    """OpenAI embeddings wrapper with consistent API."""

    def __init__(self, model_name: str = "text-embedding-3-small"):
        self.model_name = model_name
        self.model = OpenAIEmbeddings(model=self.model_name)

        # Dimension probe (safe + useful logs)
        dim_probe = self.model.embed_query("dimension probe")
        self.dim = len(dim_probe)
        print(f"[EmbeddingManager] Model={self.model_name} | dim={self.dim}")

    def embed_documents(self, texts: List[str]) -> np.ndarray:
        """Embed multiple documents (list[str]) -> (n, dim)."""
        if not texts:
            return np.empty((0, self.dim), dtype=np.float32)
        vecs = self.model.embed_documents(texts)
        return np.asarray(vecs, dtype=np.float32)

    def embed_query(self, query: str) -> np.ndarray:
        """Embed a single query -> (dim,)."""
        if not query:
            return np.zeros((self.dim,), dtype=np.float32)
        v = self.model.embed_query(query)
        return np.asarray(v, dtype=np.float32)

In [111]:
# ---------------------------------------------------------------------------
# 2) Vector store wrapper (ChromaDB)
#    - cosine space
#    - deterministic IDs + upsert
#    - metadata sanitization
# ---------------------------------------------------------------------------

SimpleMeta = Dict[str, Union[str, int, float, bool]]

class VectorStore:
    """Persistent ChromaDB collection with robust indexing."""

    def __init__(
        self,
        collection_name: str = "pdf_documents_openai_1536",
        persist_directory: str = "../data/vector_store",
        reset: bool = False,
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.reset = reset

        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        if self.reset:
            try:
                self.client.delete_collection(self.collection_name)
                print(f"[VectorStore] Deleted collection: {self.collection_name}")
            except Exception:
                pass

        # IMPORTANT: enforce cosine distance
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "PDF document embeddings for RAG",
                "hnsw:space": "cosine",  # <- critical if you interpret distance as 1 - similarity
            },
        )

        print(f"[VectorStore] Collection={self.collection_name} | count={self.collection.count()}")

    @staticmethod
    def _sanitize_metadata(md: Dict[str, Any]) -> SimpleMeta:
        """Chroma metadata must be flat simple types. Drop None; stringify complex."""
        clean: SimpleMeta = {}
        for k, v in (md or {}).items():
            if v is None:
                continue
            if isinstance(v, (str, int, float, bool)):
                clean[k] = v
            else:
                clean[k] = str(v)
        return clean

    @staticmethod
    def _normalize_source(md: Dict[str, Any]) -> str:
        src = md.get("source") or md.get("file_path") or md.get("path") or "unknown_source"
        return str(src)

    @staticmethod
    def _stable_chunk_id(source: str, page: int, chunk_index: int, content: str) -> str:
        # Hash only a prefix to keep it deterministic and short
        h = hashlib.sha256(content.encode("utf-8")).hexdigest()[:16]
        # Keep ID stable across runs
        return f"{source}::p{page}::c{chunk_index}::{h}"

    def add_documents(self, documents: List[Any], embeddings: np.ndarray) -> None:
        """
        Upsert LangChain Documents + embeddings into ChromaDB with stable IDs.
        Prevent duplicates across re-index runs.
        """
        if not documents:
            print("[VectorStore] No documents provided.")
            return
        if not isinstance(embeddings, np.ndarray) or embeddings.ndim != 2:
            raise ValueError("embeddings must be a 2D numpy array: (n_docs, dim)")
        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents must match number of embeddings")

        ids: List[str] = []
        metadatas: List[SimpleMeta] = []
        docs_text: List[str] = []
        embs_list: List[List[float]] = embeddings.astype(np.float32).tolist()

        for i, doc in enumerate(documents):
            md_raw = getattr(doc, "metadata", {}) or {}
            content = (getattr(doc, "page_content", "") or "").strip()
            if not content:
                continue

            source = self._normalize_source(md_raw)
            page = int(md_raw.get("page") or 0)

            # IMPORTANT: chunk_index should already be set during splitting (per page or global)
            chunk_index = int(md_raw.get("chunk_index") if md_raw.get("chunk_index") is not None else i)

            doc_id = self._stable_chunk_id(source, page, chunk_index, content)

            md = self._sanitize_metadata(md_raw)
            md["source"] = source
            md["page"] = page
            md["chunk_index"] = chunk_index
            md["content_length"] = len(content)

            ids.append(doc_id)
            metadatas.append(md)
            docs_text.append(content)

        if not ids:
            print("[VectorStore] Nothing to upsert (all empty?).")
            return

        self.collection.upsert(
            ids=ids,
            embeddings=embs_list,
            metadatas=metadatas,
            documents=docs_text,
        )

        print(f"[VectorStore] Upserted {len(ids)} chunks | new count={self.collection.count()}")

In [112]:
# ---------------------------------------------------------------------------
# 3) Text preparation utilities + chunking helpers
# ---------------------------------------------------------------------------

import re
from typing import Any, Dict, List, Tuple
from langdetect import detect, DetectorFactory

# Deterministic lang detection
DetectorFactory.seed = 42

SUPPORTED_LANGS = {"fr", "en", "it"}

def detect_language_safe(text: str) -> str:
    """
    Detect language for FR/EN/IT only.
    Returns: 'fr' | 'en' | 'it' | 'other' | 'unknown'
    """
    if not text:
        return "unknown"

    # langdetect is less reliable on very short strings
    compact = re.sub(r"\s+", " ", text).strip()
    if len(compact) < 40:
        return "unknown"

    try:
        lang = detect(compact)
        return lang if lang in SUPPORTED_LANGS else "other"
    except Exception:
        return "unknown"


def clean_text(text: str) -> str:
    """Clean PDF-extracted text while preserving paragraphs."""
    if not text:
        return ""

    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl").replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


def prepare_chunks(chunks: List[Any]) -> Tuple[List[Any], List[str]]:
    """
    - Drop empty chunks
    - Clean page_content
    - Ensure required metadata keys exist
    - Add metadata['lang'] = fr/en/it/other/unknown
    """
    filtered: List[Any] = []
    texts: List[str] = []

    for doc in chunks:
        raw = getattr(doc, "page_content", "") or ""
        cleaned = clean_text(raw)
        if not cleaned:
            continue

        doc.page_content = cleaned

        md = getattr(doc, "metadata", {}) or {}

        # minimal normalization
        if "source" in md:
            md["source"] = str(md["source"])

        if "page" in md:
            try:
                md["page"] = int(md["page"])
            except Exception:
                md["page"] = 0

        # ✅ language detection (FR/EN/IT)
        md["lang"] = detect_language_safe(cleaned)

        doc.metadata = md

        filtered.append(doc)
        texts.append(cleaned)

    return filtered, texts


def assign_chunk_index_per_page(chunks: List[Any]) -> List[Any]:
    """
    Ensure chunk_index is stable and meaningful:
    chunk_index is counted per (source,page) group (recommended for PDFs).
    """
    counters: Dict[Tuple[str, int], int] = {}
    for doc in chunks:
        md = getattr(doc, "metadata", {}) or {}
        src = str(md.get("source") or md.get("file_path") or "unknown_source")
        page = int(md.get("page") or 0)
        key = (src, page)
        counters[key] = counters.get(key, 0) + 1
        md["chunk_index"] = counters[key] - 1  # 0-based
        md["chunk_len"] = len(getattr(doc, "page_content", "") or "")
        doc.metadata = md
    return chunks

In [113]:
# ---------------------------------------------------------------------------
# 4) Embedding generation: batching + retries + optional cache
# ---------------------------------------------------------------------------

@dataclass
class EmbeddingConfig:
    batch_size: int = 128
    sleep_s: float = 0.25
    max_retries: int = 5
    use_cache: bool = True
    cache_path: Path = Path("./data/emb_cache_text-embedding-3-small.jsonl")


def _hash_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _load_cache(path: Path) -> Dict[str, List[float]]:
    cache: Dict[str, List[float]] = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            obj = json.loads(line)
            cache[obj["h"]] = obj["v"]
    return cache


def _append_cache(path: Path, items: List[Tuple[str, List[float]]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        for h, v in items:
            f.write(json.dumps({"h": h, "v": v}) + "\n")


def generate_embeddings_batched(
    embedding_manager: EmbeddingManager,
    texts: List[str],
    cfg: EmbeddingConfig = EmbeddingConfig(),
) -> np.ndarray:
    """Batch + retry + optional caching, returns (n, dim)."""
    if not texts:
        return np.empty((0, embedding_manager.dim), dtype=np.float32)

    cache: Dict[str, List[float]] = _load_cache(cfg.cache_path) if cfg.use_cache else {}
    vectors: List[Optional[List[float]]] = [None] * len(texts)

    missing_texts: List[str] = []
    missing_idx: List[int] = []

    for i, t in enumerate(texts):
        h = _hash_text(t)
        if cfg.use_cache and h in cache:
            vectors[i] = cache[h]
        else:
            missing_texts.append(t)
            missing_idx.append(i)

    # Embed missing only
    if missing_texts:
        new_vectors_flat: List[List[float]] = []

        for start in range(0, len(missing_texts), cfg.batch_size):
            batch = missing_texts[start:start + cfg.batch_size]

            for attempt in range(1, cfg.max_retries + 1):
                try:
                    batch_arr = embedding_manager.embed_documents(batch)  # (b, dim)
                    new_vectors_flat.extend(batch_arr.tolist())
                    break
                except Exception as e:
                    if attempt == cfg.max_retries:
                        raise
                    wait = cfg.sleep_s * (2 ** (attempt - 1))
                    print(f"[Embeddings] batch {start}-{start+len(batch)} failed (attempt {attempt}): {e}")
                    print(f"[Embeddings] retrying in {wait:.1f}s...")
                    time.sleep(wait)

            time.sleep(cfg.sleep_s)

        to_cache: List[Tuple[str, List[float]]] = []
        for t, v, i in zip(missing_texts, new_vectors_flat, missing_idx):
            vectors[i] = v
            if cfg.use_cache:
                to_cache.append((_hash_text(t), v))

        if cfg.use_cache and to_cache:
            _append_cache(cfg.cache_path, to_cache)

    final = np.asarray(vectors, dtype=np.float32)
    return final

In [114]:
# ---------------------------------------------------------------------------
# 5) End-to-end: chunks -> embeddings -> store
# ---------------------------------------------------------------------------

def embed_and_store_chunks(
    chunks: List[Any],
    embedding_manager: EmbeddingManager,
    vectorstore: VectorStore,
    cfg: EmbeddingConfig = EmbeddingConfig(),
) -> None:
    """
    Full pipeline:
      chunks -> clean/filter -> assign chunk_index -> embed -> upsert
    """
    filtered_chunks, texts = prepare_chunks(chunks)
    if not filtered_chunks:
        print("[Pipeline] No valid chunks.")
        return

    # IMPORTANT: assign chunk_index per (source,page) for stable IDs
    filtered_chunks = assign_chunk_index_per_page(filtered_chunks)

    embeddings = generate_embeddings_batched(embedding_manager, texts, cfg)
    if embeddings.shape[0] != len(filtered_chunks):
        raise RuntimeError("Embeddings/texts/chunks misalignment (should never happen).")

    print(f"[Pipeline] embeddings shape: {embeddings.shape}")
    vectorstore.add_documents(filtered_chunks, embeddings)

In [115]:
# ---------------------------------------------------------------------------
# 6) Retriever (Chroma) - robust scoring + optional filtering (HARDENED)
# ---------------------------------------------------------------------------

class RAGRetriever:
    """
    Query retriever for ChromaDB.

    Notes:
    - If collection is configured with hnsw:space=cosine:
        similarity = 1 - distance
    - If not sure about space, we still return distance, and similarity is computed
      only for cosine. (Safe default)
    """

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    @staticmethod
    def _clean_query(query: str) -> str:
        return (query or "").strip()

    @staticmethod
    def _normalize_where(where: Optional[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
        """
        Make Chroma 'where' compatible across versions:
        - keep operator queries ($and/$or/...) as is
        - if multiple simple fields -> wrap into {"$and":[...]}
        """
        if not where:
            return None

        if any(str(k).startswith("$") for k in where.keys()):
            return where

        if len(where) == 1:
            return where

        return {"$and": [{k: v} for k, v in where.items()]}

    def _is_cosine_space(self) -> bool:
        """
        Best-effort detection: if collection metadata contains hnsw:space == cosine.
        If not accessible, assume cosine (common) but keep behavior stable.
        """
        try:
            meta = getattr(self.vector_store.collection, "metadata", None)
            if isinstance(meta, dict):
                return str(meta.get("hnsw:space", "")).lower() == "cosine"
        except Exception:
            pass
        return True  # safe default for most setups

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0,
        metadata_filter: Optional[Dict[str, Any]] = None,
    ) -> List[Dict[str, Any]]:

        query = self._clean_query(query)
        if not query:
            return []

        where = self._normalize_where(metadata_filter)

        qvec = self.embedding_manager.embed_query(query).astype(np.float32).tolist()

        results = self.vector_store.collection.query(
            query_embeddings=[qvec],
            n_results=top_k,
            where=where,
        )

        docs = results.get("documents", [[]])[0] if results else []
        metas = results.get("metadatas", [[]])[0] if results else []
        dists = results.get("distances", [[]])[0] if results else []
        ids = results.get("ids", [[]])[0] if results else []

        if not docs:
            return []

        cosine_mode = self._is_cosine_space()

        out: List[Dict[str, Any]] = []
        for rank, (doc_id, content, md, dist) in enumerate(zip(ids, docs, metas, dists), start=1):
            dist_f = float(dist)

            # cosine distance -> similarity
            sim = 1.0 - dist_f if cosine_mode else None

            # If not cosine, we cannot safely threshold on similarity. So we:
            # - apply threshold only when cosine
            # - otherwise, return results without thresholding (or you can threshold on distance)
            if sim is None or sim >= score_threshold:
                out.append(
                    {
                        "id": doc_id,
                        "content": content,
                        "metadata": md,
                        "similarity_score": round(sim, 4) if sim is not None else None,
                        "distance": round(dist_f, 4),
                        "rank": rank,
                    }
                )

        if cosine_mode:
            out.sort(key=lambda x: (x["similarity_score"] is not None, x["similarity_score"]), reverse=True)

        return out

In [ ]:
# ---------------------------------------------------------------------------
# 7) Usage (Notebook)
# ---------------------------------------------------------------------------

# 0) Safety checks
if not documentspdf:
    raise ValueError("documentspdf is empty. Load your PDF documents first.")

# 1) Clean docs before split
for doc in documentspdf:
    doc.page_content = clean_text(getattr(doc, "page_content", "") or "")

# 2) Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = text_splitter.split_documents(documentspdf)
print(f"Split done: {len(chunks)} chunks")


# 🔥 AJOUTE ÇA
for c in chunks:
    c.metadata["site"] = "Couvet"
    c.metadata["doc_type"] = "SOP"

# 3) Build pipeline objects
embedding_manager = EmbeddingManager(model_name="text-embedding-3-small")
vectorstore = VectorStore(collection_name="pdf_documents_openai_1536", reset=False)

# 4) Index
embed_and_store_chunks(chunks, embedding_manager, vectorstore)

# Quick sanity check (optional)
print("Collection count:", vectorstore.collection.count())
try:
    sample = vectorstore.collection.peek(2)
    print("Peek metadatas:", sample.get("metadatas"))
except Exception as e:
    print("Peek not available in this Chroma version:", e)

# 5) Retrieve (no filter)
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

res = rag_retriever.retrieve(
    " c'est quoi la sop it ? ",
    top_k=5,
    score_threshold=0.38,
)
res[:2]

In [117]:
from typing import Any, Dict, List, Tuple

def dedup_results_by_source_page(
    results: List[Dict[str, Any]],
    max_per_source: int = 3,
    max_per_page: int = 1,
) -> List[Dict[str, Any]]:
    """
    Keep only a limited number of chunks per source and per page.
    - max_per_page=1: avoids duplicates from same page
    - max_per_source=3: avoids one PDF dominating everything
    """
    out: List[Dict[str, Any]] = []
    per_source = {}
    per_page = {}

    for r in results:
        md = r.get("metadata", {}) or {}
        source = str(md.get("source", "unknown_source"))
        page = int(md.get("page", 0) or 0)

        page_key = (source, page)

        per_source[source] = per_source.get(source, 0)
        per_page[page_key] = per_page.get(page_key, 0)

        if per_source[source] >= max_per_source:
            continue
        if per_page[page_key] >= max_per_page:
            continue

        out.append(r)
        per_source[source] += 1
        per_page[page_key] += 1

    return out

In [118]:
import re
from typing import Any, Dict, List, Set

def _tokenize(text: str) -> Set[str]:
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9àâäçéèêëîïôöùûüœæ'\s-]", " ", text)
    tokens = set(t for t in text.split() if len(t) >= 3)
    return tokens

def _jaccard(a: Set[str], b: Set[str]) -> float:
    if not a or not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0

def mmr_lite(
    results: List[Dict[str, Any]],
    k: int = 5,
    lambda_mult: float = 0.75,
) -> List[Dict[str, Any]]:
    """
    Select k results balancing:
    - relevance: similarity_score
    - diversity: 1 - jaccard(text_tokens)

    lambda_mult in [0..1]:
    - 1.0 -> only relevance (no diversity)
    - 0.0 -> only diversity
    """
    if not results:
        return []

    # sort by relevance first (good baseline)
    candidates = sorted(results, key=lambda x: x.get("similarity_score", 0), reverse=True)

    selected: List[Dict[str, Any]] = []
    selected_tokens: List[Set[str]] = []

    # precompute tokens
    cand_tokens = [_tokenize(r.get("content", "")) for r in candidates]

    while candidates and len(selected) < k:
        best_idx = 0
        best_score = -1e9

        for i, r in enumerate(candidates):
            rel = float(r.get("similarity_score", 0.0))

            if not selected:
                mmr_score = rel
            else:
                # similarity to already selected (max redundancy)
                redund = max(_jaccard(cand_tokens[i], t) for t in selected_tokens) if selected_tokens else 0.0
                div = 1.0 - redund
                mmr_score = lambda_mult * rel + (1.0 - lambda_mult) * div

            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = i

        chosen = candidates.pop(best_idx)
        chosen_tok = cand_tokens.pop(best_idx)

        selected.append(chosen)
        selected_tokens.append(chosen_tok)

    return selected

In [146]:
from typing import Any, Dict, List, Optional

def retrieve_postprocessed(
    rag_retriever,
    query: str,
    top_k_retrieve: int = 12,      # on récupère plus large au début
    final_k: int = 5,              # puis on réduit après post-processing
    score_threshold: float = 0.2,
    metadata_filter: Optional[Dict[str, Any]] = None,
    max_per_source: int = 3,
    max_per_page: int = 1,
    lambda_mult: float = 0.75,
) -> List[Dict[str, Any]]:

    raw = rag_retriever.retrieve(
        query,
        top_k=top_k_retrieve,
        score_threshold=score_threshold,
        metadata_filter=metadata_filter,
    )

    # 1) Dedup
    deduped = dedup_results_by_source_page(
        raw,
        max_per_source=max_per_source,
        max_per_page=max_per_page,
    )

    # 2) MMR lite (diverse top-k)
    diversified = mmr_lite(
        deduped,
        k=final_k,
        lambda_mult=lambda_mult,
    )

    return diversified

In [ ]:
results = retrieve_postprocessed(
    rag_retriever,
    "backup validation procedure",
    top_k_retrieve=15,
    final_k=5,
    score_threshold=0.2,
    lambda_mult=0.75,
)

results[:2]

In [148]:
from typing import Any, Dict, List

def build_context_with_citations(
    results: List[Dict[str, Any]],
    max_chars_per_chunk: int = 1200,
) -> str:
    """
    Build a single context string from retrieved chunks.
    Each chunk is preceded by a citation header the LLM can reuse.
    """
    blocks = []
    for i, r in enumerate(results, start=1):
        md = r.get("metadata", {}) or {}
        source = str(md.get("source", "unknown_source"))
        page = md.get("page", "?")
        chunk_index = md.get("chunk_index", "?")
        score = r.get("similarity_score", None)

        text = (r.get("content") or "").strip()
        if len(text) > max_chars_per_chunk:
            text = text[:max_chars_per_chunk] + "..."

        citation = f"[C{i} | {source} | p.{page} | chunk {chunk_index}]"

        blocks.append(
            f"{citation}\n{text}"
        )

    return "\n\n---\n\n".join(blocks)

In [149]:
SYSTEM_PROMPT = """You are an internal documentation assistant.
Rules:
- Use ONLY the provided CONTEXT.
- If the CONTEXT does not contain enough information, answer exactly:
  "Insufficient evidence in provided documents."
- Do NOT invent procedures, steps, numbers, or compliance statements.
- Every key statement must include at least one citation tag like [C1] or [C2].
"""

def build_user_prompt(question: str, context: str) -> str:
    return f"""CONTEXT:
{context}

QUESTION:
{question}

OUTPUT FORMAT:
- Provide a clear answer.
- Use bullet points for procedures/steps.
- Add citations like [C1] after the statements they support.
- If insufficient: "Insufficient evidence in provided documents."
"""

In [150]:
# 1) Retrieve + post-process (recommended)
results = retrieve_postprocessed(
    rag_retriever,
    "What is the procedure for backup validation?",
    top_k_retrieve=15,
    final_k=6,
    score_threshold=0.25,
    lambda_mult=0.75,
)

# 2) Build context
context = build_context_with_citations(results)

print(context[:1500])  # preview

[C1 | ..\data\pdf_files\SOP-IT-0002_Backup_Restore_Validation.pdf | p.0 | chunk 0]
SOP-IT-0002 — Backup & Restore Validation
Procedure
Document ID: SOP-IT-0002
Title: Backup & Restore Validation Procedure (GxP Systems)
Version: 1.0
Status: Effective
Site: Couvet
Department: IT
Effective Date: 2026-03-01
1. Purpose
Define the procedure for backup validation and restore testing for GxP computerized systems to
ensure data integrity, availability, and compliance with EU GMP Annex 11 and 21 CFR Part 11.
2. Scope
Applies to all GxP systems hosted or managed by IT including servers, databases, applications and
storage systems.
3. Backup Validation Requirements
- Verification of successful backup jobs.
- Verification that all required components are included.
- Integrity checks (checksum/hash where applicable).
- Verification of retention policy compliance.
4. Restore Test Procedure

---

[C2 | ..\data\pdf_files\SOP-IT-001_Backup_Validation_Procedure.pdf | p.0 | chunk 0]
Standard Operating Pro

In [151]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",   # tu peux changer
    temperature=0.0
)

In [152]:
def generate_answer_with_citations(question: str, context: str) -> str:
    user_prompt = build_user_prompt(question, context)

    resp = llm.invoke([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])

    return resp.content

In [175]:
def rag_answer(question: str) -> Dict[str, Any]:
    results = retrieve_postprocessed(
        rag_retriever,
        question,
        top_k_retrieve=15,
        final_k=6,
        score_threshold=0.25,
        lambda_mult=0.75,
    )

    if not results:
        return {"answer": "Insufficient evidence in provided documents.", "results": [], "context": ""}

    context = build_context_with_citations(results, max_chars_per_chunk=1200)
    answer = generate_answer_with_citations(question, context)

    return {"answer": answer, "results": results, "context": context}

In [176]:
out = rag_answer("What is the procedure for backup validation?")
print(out["answer"])

The procedure for backup validation includes the following steps:

- Verification of successful backup jobs.
- Verification that all required components are included in the backup.
- Performing integrity checks such as checksum or hash where applicable.
- Verification of compliance with the retention policy [C1].

Additional related activities include backup scheduling, restore validation, and monitoring controls to ensure compliance with EU GMP Annex 11 [C2].


In [195]:
from typing import Any, Dict, List, Tuple

def extract_citations(results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Build a clean, deduplicated citations list from retrieval results.
    Dedup by (source, page).
    """
    citations: List[Dict[str, Any]] = []
    seen = set()

    for i, r in enumerate(results, start=1):
        md = r.get("metadata", {}) or {}
        source = str(md.get("source", "unknown_source"))
        page = md.get("page", None)
        chunk_index = md.get("chunk_index", None)
        score = r.get("similarity_score", None)

        # display page in human-friendly way if 0-based
        page_display = page
        if isinstance(page, int):
            page_display = page + 1  # convert 0->1

        key = (source, page_display)
        if key in seen:
            continue
        seen.add(key)

        citations.append(
            {
                "ref": f"C{i}",
                "source": source,
                "page": page_display,
                "chunk_index": chunk_index,
                "similarity_score": score,
            }
        )

    return citations

In [178]:
import numpy as np
from typing import Any, Dict, List, Optional

def compute_confidence(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Compute confidence from similarity_score values (assumed in [0..1]).
    Returns: {"score": float, "label": str, "details": {...}}
    """
    scores = [r.get("similarity_score") for r in results if isinstance(r.get("similarity_score"), (int, float))]
    if not scores:
        return {"score": 0.0, "label": "none", "details": {"reason": "no_scores"}}

    scores = np.array(scores, dtype=float)
    scores = np.clip(scores, 0.0, 1.0)

    top1 = float(scores[0])
    top3_mean = float(np.mean(scores[: min(3, len(scores))]))
    n = len(scores)

    # Weighted confidence
    conf = 0.65 * top1 + 0.35 * top3_mean

    # small penalty if too few results
    if n == 1:
        conf *= 0.88
    elif n == 2:
        conf *= 0.95

    conf = float(np.clip(conf, 0.0, 1.0))

    if conf >= 0.60:
        label = "high"
    elif conf >= 0.40:
        label = "medium"
    else:
        label = "low"

    return {
        "score": round(conf, 4),
        "label": label,
        "details": {
            "top1": round(top1, 4),
            "top3_mean": round(top3_mean, 4),
            "n_results_used": n,
        },
    }

In [196]:
from typing import Any, Dict

def rag_answer_with_meta(
    question: str,
    rag_retriever,
    top_k_retrieve: int = 15,
    final_k: int = 6,
    score_threshold: float = 0.25,
    lambda_mult: float = 0.75,
    metadata_filter=None,
    return_debug: bool = False,
) -> Dict[str, Any]:

    results = retrieve_postprocessed(
        rag_retriever,
        question,
        top_k_retrieve=top_k_retrieve,
        final_k=final_k,
        score_threshold=score_threshold,
        lambda_mult=lambda_mult,
        metadata_filter=metadata_filter,
    )

    confidence = compute_confidence(results)
    citations = extract_citations(results)

    if not results:
        payload = {
            "answer": "Insufficient evidence in provided documents.",
            "citations": [],
            "confidence": confidence,
        }
        if return_debug:
            payload.update({"results": [], "context": ""})
        return payload

    context = build_context_with_citations(results, max_chars_per_chunk=1200)
    answer = generate_answer_with_citations(question, context)

    payload = {
        "answer": answer,
        "citations": citations,
        "confidence": confidence,
    }

    if return_debug:
        payload.update({"results": results, "context": context})

    return payload

In [197]:
from typing import Any, Dict

def rag_answer_with_meta(
    question: str,
    rag_retriever,
    top_k_retrieve: int = 15,
    final_k: int = 6,
    score_threshold: float = 0.25,
    lambda_mult: float = 0.75,
    metadata_filter=None,
    return_debug: bool = False,
) -> Dict[str, Any]:

    results = retrieve_postprocessed(
        rag_retriever,
        question,
        top_k_retrieve=top_k_retrieve,
        final_k=final_k,
        score_threshold=score_threshold,
        lambda_mult=lambda_mult,
        metadata_filter=metadata_filter,
    )

    confidence = compute_confidence(results)
    citations = extract_citations(results)

    if not results:
        payload = {
            "answer": "Insufficient evidence in provided documents.",
            "citations": [],
            "confidence": confidence,
        }
        if return_debug:
            payload.update({"results": [], "context": ""})
        return payload

    context = build_context_with_citations(results, max_chars_per_chunk=1200)
    answer = generate_answer_with_citations(question, context)

    payload = {
        "answer": answer,
        "citations": citations,
        "confidence": confidence,
    }

    if return_debug:
        payload.update({"results": results, "context": context})

    return payload

In [207]:
out = rag_answer_with_meta(
    "What is the procedure for backup validation? ",
    rag_retriever,
    return_debug=False
)

print(out["answer"])
print("\nCITATIONS:")
for c in out["citations"]:
    print(c)

print("\nCONFIDENCE:", out["confidence"])

The procedure for backup validation includes the following steps:

- Verification of successful backup jobs.
- Verification that all required components are included in the backup.
- Performing integrity checks such as checksum or hash where applicable.
- Verification of compliance with the retention policy [C1].

Additional related activities include backup scheduling, restore validation, and monitoring controls to ensure compliance with EU GMP Annex 11 [C2].

CITATIONS:
{'ref': 'C1', 'source': '..\\data\\pdf_files\\SOP-IT-0002_Backup_Restore_Validation.pdf', 'page': 1, 'chunk_index': 0, 'similarity_score': 0.6744}
{'ref': 'C2', 'source': '..\\data\\pdf_files\\SOP-IT-001_Backup_Validation_Procedure.pdf', 'page': 1, 'chunk_index': 0, 'similarity_score': 0.585}
{'ref': 'C3', 'source': '..\\data\\pdf_files\\SOP-IT-0002_Backup_Restore_Validation.pdf', 'page': 2, 'chunk_index': 0, 'similarity_score': 0.4489}
{'ref': 'C4', 'source': '..\\data\\pdf_files\\SOP-IT-0003_Backup_Monitoring.pdf', 

In [182]:
import os
from typing import Any, Dict, List

def friendly_source_label(md: Dict[str, Any]) -> str:
    """
    Return a user-friendly label without technical paths.
    Priority: doc_id -> file_name -> basename(source) -> 'Document'
    """
    if not md:
        return "Document"

    if md.get("doc_id"):
        return str(md["doc_id"])

    if md.get("file_name"):
        return str(md["file_name"])

    src = md.get("source", "")
    if src:
        return os.path.basename(str(src))

    return "Document"

In [199]:
def extract_citations_friendly(results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Clean citations for end-users:
    - no filesystem path
    - no chunk_index
    - no similarity score
    - dedup by (label, page)
    """
    citations = []
    seen = set()

    for i, r in enumerate(results, start=1):
        md = r.get("metadata", {}) or {}
        label = friendly_source_label(md)

        page = md.get("page", None)
        page_display = page + 1 if isinstance(page, int) else page  # 0->1 if needed

        title = md.get("doc_title", None)  # optional

        key = (label, page_display)
        if key in seen:
            continue
        seen.add(key)

        item = {
            "ref": f"C{i}",
            "document": label,
            "page": page_display,
        }
        if title:
            item["title"] = str(title)

        citations.append(item)

    return citations

In [200]:
def format_sources_for_users(citations: List[Dict[str, Any]]) -> str:
    lines = []
    for c in citations:
        title_part = f" — {c['title']}" if "title" in c else ""
        page_part = f"p.{c['page']}" if c.get("page") not in (None, "?") else "page ?"
        lines.append(f"- [{c['ref']}] {c['document']}{title_part} ({page_part})")
    return "\n".join(lines)

In [201]:
out = rag_answer_with_meta(
    "What is the procedure for backup validation?",
    rag_retriever,
    return_debug=True
)

print(out["answer"])
print("\nSOURCES:")
citations_clean = extract_citations_friendly(out["results"])
print(format_sources_for_users(citations_clean))
print("\nCONFIDENCE:", out["confidence"])

The procedure for backup validation includes the following steps:

- Verification of successful backup jobs.
- Verification that all required components are included.
- Integrity checks such as checksum or hash where applicable.
- Verification of retention policy compliance [C1].

Additional related activities include backup scheduling, restore validation, and monitoring controls to ensure compliance with EU GMP Annex 11 [C2].

SOURCES:
- [C1] SOP-IT-0002_Backup_Restore_Validation.pdf (p.1)
- [C2] SOP-IT-001_Backup_Validation_Procedure.pdf (p.1)
- [C3] SOP-IT-0002_Backup_Restore_Validation.pdf (p.2)
- [C4] SOP-IT-0003_Backup_Monitoring.pdf (p.1)
- [C5] WI-IT-0101_Restore_Test_Work_Instruction.pdf (p.1)
- [C6] SOP-IT-0003_Backup_Monitoring.pdf (p.2)

CONFIDENCE: {'score': 0.6378, 'label': 'high', 'details': {'top1': 0.6745, 'top3_mean': 0.5695, 'n_results_used': 6}}


In [186]:
SYSTEM_PROMPT_PUBLIC = """You are a helpful assistant.

You may use TWO sources of information:
1) PROVIDED CONTEXT (documents)
2) GENERAL KNOWLEDGE (best practices)

Rules:
- If the CONTEXT contains relevant information, prioritize it and cite with [C1], [C2], etc.
- If the CONTEXT is insufficient, you may answer using general knowledge.
- When you use general knowledge, clearly label it as "General guidance" and do NOT invent document citations.
- Never claim that a statement comes from the documents unless it is supported by the CONTEXT.
- Keep answers practical and structured.
"""

In [187]:
def build_user_prompt_public(question: str, context: str) -> str:
    return f"""CONTEXT (may be empty or insufficient):
{context}

QUESTION:
{question}

OUTPUT FORMAT:
1) Answer (clear and actionable)
2) If you used the context: add citations [C#] next to supported statements
3) If context is insufficient: include a section "General guidance" with best practices
4) End with a short list: "What I would need from your documents to be 100% sure" (max 3 bullets)
"""

In [188]:
from typing import Any, Dict

def rag_answer_public(
    question: str,
    rag_retriever,
    top_k_retrieve: int = 15,
    final_k: int = 6,
    score_threshold: float = 0.20,
    lambda_mult: float = 0.75,
    metadata_filter=None,
    fallback_conf_threshold: float = 0.35,
) -> Dict[str, Any]:

    results = retrieve_postprocessed(
        rag_retriever,
        question,
        top_k_retrieve=top_k_retrieve,
        final_k=final_k,
        score_threshold=score_threshold,
        lambda_mult=lambda_mult,
        metadata_filter=metadata_filter,
    )

    confidence = compute_confidence(results)

    # Build context even if weak; model will decide what it can cite
    context = build_context_with_citations(results)

    # Decide if we likely have enough evidence to be doc-grounded
    doc_grounded = (len(results) >= 2) and (confidence["score"] >= fallback_conf_threshold)

    # Generate
    user_prompt = build_user_prompt_public(question, context)

    resp = llm.invoke([
        {"role": "system", "content": SYSTEM_PROMPT_PUBLIC},
        {"role": "user", "content": user_prompt},
    ])
    answer = resp.content

    # User-friendly citations list only if doc-grounded
    citations = extract_citations_sop_version(results) if doc_grounded else []

    return {
        "answer": answer,
        "doc_grounded": doc_grounded,
        "confidence": confidence,
        "citations": citations,
    }

In [189]:
import os
import re
from typing import Any, Dict, List, Optional

_DOCID_RE = re.compile(r"\b(SOP-[A-Z]{2,}-\d{4}|SOP-\d{4}|WI-[A-Z]{2,}-\d{4}|WI-\d{4})\b", re.IGNORECASE)
_VER_RE = re.compile(r"(?:\bv|\bver|\bversion)\s*[:=_-]?\s*(\d+(?:\.\d+)*)", re.IGNORECASE)

def _basename_from_source(md: Dict[str, Any]) -> str:
    src = md.get("source") or md.get("file_name") or ""
    return os.path.basename(str(src))

def extract_doc_id(md: Dict[str, Any]) -> Optional[str]:
    if md.get("doc_id"):
        return str(md["doc_id"])
    name = _basename_from_source(md)
    m = _DOCID_RE.search(name)
    return m.group(1).upper() if m else None

def extract_version(md: Dict[str, Any]) -> Optional[str]:
    if md.get("version"):
        return str(md["version"])
    name = _basename_from_source(md)
    m = _VER_RE.search(name)
    return m.group(1) if m else None

def extract_citations_sop_version(results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    citations = []
    seen = set()

    for i, r in enumerate(results, start=1):
        md = r.get("metadata", {}) or {}

        doc_id = extract_doc_id(md) or "DOC-UNKNOWN"
        version = extract_version(md) or "?"
        page = md.get("page", None)

        page_display = page + 1 if isinstance(page, int) else page

        key = (doc_id, version, page_display)
        if key in seen:
            continue
        seen.add(key)

        citations.append({
            "ref": f"C{i}",
            "doc_id": doc_id,
            "version": version,
            "page": page_display,
        })

    return citations

In [202]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.0
)

out = rag_answer_public("c'est quoi un backup ?", rag_retriever)
print(out["answer"])
print("\nDoc-grounded:", out["doc_grounded"])
print("Confidence:", out["confidence"])
if out["citations"]:
    print("\nSOURCES:")
    for c in out["citations"]:
        print(c)

1) Answer:  
Un backup, ou sauvegarde, est une copie de données informatiques réalisée afin de pouvoir restaurer ces données en cas de perte, de corruption ou d'incident affectant les données originales. Il permet d'assurer la protection continue des informations en garantissant leur disponibilité et intégrité, notamment dans des environnements critiques comme les systèmes GxP en industrie pharmaceutique.

2) Context used:  
Les documents mentionnent la surveillance des jobs de sauvegarde, la validation des sauvegardes, la restauration à partir des sauvegardes, ainsi que la conformité aux exigences réglementaires (EU GMP Annex 11, 21 CFR Part 11) pour garantir l'intégrité et la disponibilité des données [C1], [C2], [C3].

3) General guidance:  
Un backup est une pratique standard en informatique qui consiste à copier des données à un moment donné pour pouvoir les récupérer en cas de défaillance du système, d'erreur humaine, d'attaque malveillante ou de sinistre. Les sauvegardes doivent

In [203]:
from typing import Any, Dict

def rag_answer(
    question: str,
    rag_retriever,
    mode: str = "personal",
    top_k_retrieve: int = 15,
    final_k: int = 6,
    score_threshold: float = 0.20,
    lambda_mult: float = 0.75,
    metadata_filter=None,
    fallback_conf_threshold: float = 0.35,
) -> Dict[str, Any]:

    results = retrieve_postprocessed(
        rag_retriever,
        question,
        top_k_retrieve=top_k_retrieve,
        final_k=final_k,
        score_threshold=score_threshold,
        lambda_mult=lambda_mult,
        metadata_filter=metadata_filter,
    )

    confidence = compute_confidence(results)

    # doc-grounded decision
    doc_grounded = (len(results) >= 2) and (confidence["score"] >= fallback_conf_threshold)

    context = build_context_with_citations(results)

    if mode == "personal":
        SYSTEM_PROMPT_PERSONAL = """You are a helpful assistant.

You may use:
1) Provided CONTEXT (documents)
2) Your general knowledge (ChatGPT)

Rules:
- If context is strong and relevant, use it and cite with [C#].
- If context is weak or insufficient, answer using general knowledge.
- NEVER invent citations.
- If using general knowledge, do not add citation tags.
- Keep answer clear and structured.
"""

        user_prompt = f"""CONTEXT:
{context}

QUESTION:
{question}
"""

        resp = llm.invoke([
            {"role": "system", "content": SYSTEM_PROMPT_PERSONAL},
            {"role": "user", "content": user_prompt},
        ])

        answer = resp.content

        # ✅ sources handling
        if doc_grounded:
            citations = extract_citations_generic(results)
            source_type = "documents"
        else:
            citations = []
            source_type = "chatgpt"
            # optional: prepend a short disclaimer for the user
            answer = "⚠️ Source: ChatGPT (general knowledge — not found in your documents).\n\n" + answer

        return {
            "answer": answer,
            "mode": "personal",
            "source_type": source_type,      # "documents" or "chatgpt"
            "doc_grounded": doc_grounded,
            "confidence": confidence,
            "citations": citations,
        }

    # strict enterprise
    else:
        if not doc_grounded:
            return {
                "answer": "Insufficient evidence in provided documents.",
                "mode": "enterprise",
                "source_type": "documents",
                "doc_grounded": False,
                "confidence": confidence,
                "citations": [],
            }

        resp = llm.invoke([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(question, context)},
        ])

        return {
            "answer": resp.content,
            "mode": "enterprise",
            "source_type": "documents",
            "doc_grounded": True,
            "confidence": confidence,
            "citations": extract_citations_generic(results),
        }

In [204]:
def extract_citations_generic(results):
    citations = []
    seen = set()

    for i, r in enumerate(results, start=1):
        md = r.get("metadata", {}) or {}

        # 🔥 Ignore author completely
        label = (
            md.get("doc_title")
            or md.get("doc_id")
            or os.path.basename(str(md.get("source", "")))
            or "Document"
        )

        page = md.get("page")
        page_display = page + 1 if isinstance(page, int) else page

        key = (label, page_display)
        if key in seen:
            continue
        seen.add(key)

        citations.append({
            "ref": f"C{i}",
            "document": label,
            "page": page_display if page_display is not None else "—",
        })

    return citations

In [205]:
from typing import List, Dict, Any

def format_sources_generic(citations: List[Dict[str, Any]]) -> str:
    """
    Clean user-friendly display of sources.
    Works for any document type.
    """
    lines = []

    for c in citations:
        page = c.get("page")
        if page in (None, "—", "?"):
            page_text = "no page"
        else:
            page_text = f"p.{page}"

        lines.append(f"- [{c['ref']}] {c['document']} ({page_text})")

    return "\n".join(lines)

In [206]:
out = rag_answer("c'est quoi un backup ?", rag_retriever, mode="personal")
print(out["answer"])
print("\nSOURCE TYPE:", out["source_type"])

if out["source_type"] == "documents" and out["citations"]:
    print("\nSOURCES:")
    print(format_sources_generic(out["citations"]))

Un backup, ou sauvegarde en français, est une copie de données informatiques réalisée afin de pouvoir restaurer ces données en cas de perte, de corruption, ou de défaillance du système d'origine. Le but principal d'un backup est d'assurer la protection continue des données en permettant leur récupération rapide et fiable, garantissant ainsi la disponibilité et l'intégrité des informations critiques.

Dans le contexte des procédures IT décrites, le backup fait partie d'un processus contrôlé qui inclut la surveillance des tâches de sauvegarde, la validation des sauvegardes, ainsi que la réalisation de tests de restauration pour s'assurer que les données sauvegardées peuvent être récupérées efficacement en cas de besoin. Ces activités sont souvent encadrées par des normes et réglementations telles que l'EU GMP Annex 11 et le 21 CFR Part 11, particulièrement dans les environnements réglementés comme l'industrie pharmaceutique.

SOURCE TYPE: documents

SOURCES:
- [C1] SOP-IT-0003_Backup_Mon